# Validate original models against test data from Ciara

There isn't enough human-transcribed data to train against, but we can use it for validating the AI transcription models

This notebook documents the validation against the original models with no fine-tuning.

In [2]:
import subprocess

# Define the models to be used for validation.
# Each model is associated with a name, the model slug for the CLI, batch size, and number of shards.
# Name, model slug, batch size, no. of shards
MODEL_JOBS = [
    (
        "SmolVLM",
        "smolvlm2",
        50,
        1,
    ),
    (
        "Granite4",
        "granite4",
        50,
        1,
    ),
    (
        "Gemma3",
        "gemma3",
        50,
        1,
    ),
    (
        "Gemma4",
        "gemma4",
        50,
        1,
    ),
    (
        "Ministral",
        "ministral",
        15,
        1,
    ),
]

## 1. Run the extractions

The code cell below submits one extraction job for each of the five models.
It relies on the MODEL_JOBS dictionary defined in the cell above.

There are only 64 images in the test set, so these jobs will run in 15 mins or so (+queueing time).

In [ ]:
for model_name, model, batch_size, total_shards, in MODEL_JOBS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../scripts/aml_submit.sh",
            "--model",
            model,
            "--images-path",
            "test_data/from_Ciara/images",
            "--transcriptions-path",
            "documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_0",
            "--total-shards",
            str(total_shards),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

When the jobs have been submitted, the extraction registry will contain the run names we need to download and analyze the extractions.

Run the cell below to discover run names from the registry.
The run names will be automatically detected and used in subsequent cells.

In [3]:
import json
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Discover run names from extraction registry after submissions complete.

def _first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

REGISTRY_PATH = _first_existing_path(
    [Path("outputs/extraction_registry.json"), Path("../outputs/extraction_registry.json")]
 )
TARGET_IMAGES_PATH = "test_data/from_Ciara/images"

def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")

def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])

RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(TARGET_IMAGES_PATH)

for model_name, model, _batch_size, _total_shards in MODEL_JOBS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("model_slug", "")) == model
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"] )
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    print(f"\nWarning: Missing extraction runs in registry for: {', '.join(missing)}")
    print("Run the extraction submission cell first, then re-run this cell.")
else:
    print(f"\nFound all {len(RUN_NAMES)} run names.")

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]

SmolVLM: 20260616-142852
Granite4: 20260616-144739
Gemma3: 20260616-145253
Gemma4: 20260616-145335
Ministral: 20260616-145351

Found all 5 run names.


## 2. Download the extractions

When the jobs have completed successfully, download the extractions so we can analyze them locally.

In [ ]:
for run_name in RUN_NAMES:
    print(f"Downloading {run_name}...")
    subprocess.run(
        ["bash", "../scripts/aml_download.sh", "--run-name", run_name],
        check=True,
    )

## 3. Make consensus and validate

Make a consensus from the downloaded extractions, and compare with the ground-truth human-transcribed values for each image.

Run the code cell below to build consensus and validation figures.

This produces a consensus in `outputs/consensus_Ciara_data/consensus_0th_order` with transcriptions, summary, and validation figures for all cases.

In [4]:
cmd = [
    "bash",
    "../scripts/run_consensus_pipeline.sh",
    "--dataset-root",
    "../outputs/consensus_Ciara_data",
    "--variant-name",
    "consensus_0th_order",
    "--threshold",
    "3",  # minimum number of transcriptions required to form a consensus
    "--null-threshold",
    "5", # minimum number of transcriptions required to form a consensus for null
    "--precision",
    "3",
]

for extraction_dir in EXTRACTION_DIRS:
    cmd.extend(["--extraction-dir", extraction_dir])

cmd.extend(
    [
        "--validate",
        "--ground-truth-dir",
        "../test_data/from_Ciara/transcriptions",
        "--validation-sample-denominator",
        "1",
    ]
)

subprocess.run(cmd, check=True)

Error: extraction directory not found: ../outputs/extractions/20260616-142852


CalledProcessError: Command '['bash', '../scripts/run_consensus_pipeline.sh', '--dataset-root', '../outputs/consensus_Ciara_data', '--variant-name', 'consensus_0th_order', '--threshold', '3', '--null-threshold', '5', '--precision', '3', '--extraction-dir', '../outputs/extractions/20260616-142852', '--extraction-dir', '../outputs/extractions/20260616-144739', '--extraction-dir', '../outputs/extractions/20260616-145253', '--extraction-dir', '../outputs/extractions/20260616-145335', '--extraction-dir', '../outputs/extractions/20260616-145351', '--validate', '--ground-truth-dir', '../test_data/from_Ciara/transcriptions', '--validation-sample-denominator', '1']' returned non-zero exit status 1.